# Echoshader Element Examples

This notebook demonstrates the visualization elements provided by Echoshader.

The examples are intentionally focused on plotting rather than application state. Echoshader returns HoloViews, GeoViews, Plotly, or PyVista objects that can then be customized with the native options of those libraries.

The first part follows the Echopype raw-data workflow and exercises the different vertical geometries supported by the echogram element. The second part uses a processed survey dataset to demonstrate the remaining visualization elements: track, histogram, table, and curtain.


## Import packages


In [ ]:
import holoviews as hv
import numpy as np
import panel as pn
import xarray as xr

import echopype as ep
import echoshader
import geoviews as gv

from echopype.commongrid import resample_to_geometry
from echopype.qc import coerce_increasing_time, exist_reversed_time

In [ ]:
pn.extension()
hv.extension("bokeh")
gv.extension("bokeh")

We define the ek500 colormap.

In [ ]:
import matplotlib as mpl
import echopype.colormap

ek500_cmap = mpl.colormaps["ep.ek500"]

# Echogram elements

## Convert raw data

We use the same EK80 example file as the Echopype Getting Started example.


In [ ]:
raw_path = (
    "s3://noaa-wcsd-pds/data/raw/Bell_M._Shimada/SH2306/EK80/"
    "Hake-D20230811-T165727.raw"
)

ed = ep.open_raw(
    raw_path,
    sonar_model="EK80",
    storage_options={"anon": True},
)

ed

## Compute Sv

Calibrated volume backscattering strength (`Sv`) is computed from the narrowband power data.


In [ ]:
ds_Sv = ep.calibrate.compute_Sv(
    ed,
    waveform_mode="CW",
    encode_mode="power",
)

if exist_reversed_time(ds_Sv, "ping_time"):
    coerce_increasing_time(ds_Sv)

ds_Sv


## Add depth

`add_depth()` adds the physical depth associated with each acoustic sample. At this stage, `Sv` is still indexed by `range_sample`, while depth can vary with channel and ping.


In [ ]:
ds_Sv = ep.consolidate.add_depth(
    ds_Sv,
    ed,
    depth_offset=9.8,
)

selected_channel = ds_Sv.channel.sel(
    channel=ds_Sv.frequency_nominal == 200000
).item()

ds_Sv


### Single-frequency echogram

A single channel is displayed with `eshader.echogram()`. Plot styling is applied with native HoloViews options.


In [ ]:
ds_Sv.eshader.echogram(
    channel=selected_channel,
    vert_dim="depth",
).opts(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=1000,
    height=450,
)


## Compute MVBS

Binning onto a common depth and time grid produces a regular one-dimensional `depth` coordinate.


In [ ]:
ds_MVBS = ep.commongrid.compute_MVBS(
    ds_Sv,
    range_var="depth",
    range_bin="1m",
    ping_time_bin="5s",
    range_var_max="500m",
)

ds_MVBS


### Multi-frequency echograms

Pass multiple channels to create a vertical layout of echograms.
Customize their appearance using native HoloViews `.opts()`.

In [ ]:
channels = (
    ds_MVBS.sortby("frequency_nominal").channel.values.tolist()
)

display_options = dict(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=1000,
    height=250,
)

ds_MVBS.eshader.echogram(
    channel=channels,
    vert_dim="depth",
).opts(
    hv.opts.Image(**display_options),
    hv.opts.QuadMesh(**display_options),
)

## Tricolor echogram

`tricolor_echogram()` combines three channels into RGB components. Here 120 kHz is mapped to red, 38 kHz to green, and 18 kHz to blue.


In [ ]:
rgb_channels = [
    ds_MVBS.channel.sel(
        channel=ds_MVBS.frequency_nominal == frequency
    ).item()
    for frequency in (120000, 38000, 18000)
]

ds_MVBS.eshader.tricolor_echogram(
    channel=rgb_channels,
    vmin=-80,
    vmax=-30,
    vert_dim="depth",
).opts(
    width=1000,
    height=450,
)


## Resampled vertical geometries

Echopype’s `resample_to_geometry()` aligns channels to a reference
channel or a custom range grid. The examples below display these
outputs using the same `eshader.echogram()` interface.

For details, see the
[Echopype resampling notebook](https://echopype-examples.readthedocs.io/en/latest/resample_to_geometry.html).

### Match a reference channel


In [ ]:
aligned_ds_200 = resample_to_geometry(
    ds_Sv.chunk(
        {
            "channel": 1,
            "ping_time": 1000,
            "range_sample": -1,
        }
    ),
    target_variable="Sv",
    is_log=True,
    target_channel=selected_channel,
)

aligned_ds_200 = ep.consolidate.add_depth(
    aligned_ds_200
)

aligned_ds_200.eshader.echogram(
    channel=selected_channel,
    vert_dim="depth",
).opts(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=1000,
    height=450,
)


### Custom regular target geometry

A one-dimensional target grid is applied to every ping. Echopype retains the general `range_sample` representation, while Echoshader detects that the effective vertical geometry is shared across pings.


In [ ]:
reference_range = (
    ds_Sv["echo_range"]
    .sel(channel=selected_channel)
    .isel(ping_time=0)
    .dropna("range_sample")
)

spacing = float(
    reference_range
    .diff("range_sample")
    .median()
    .compute()
)

max_range = float(
    ds_Sv["echo_range"]
    .max(skipna=True)
    .compute()
)

target_range = np.arange(
    0,
    max_range + spacing,
    spacing,
)

target_grid = xr.DataArray(
    target_range,
    dims=("range_sample",),
    coords={
        "range_sample": np.arange(target_range.size),
    },
    name="echo_range",
    attrs=ds_Sv["echo_range"].attrs,
)

aligned_ds_custom = resample_to_geometry(
    ds_Sv.chunk(
        {
            "channel": 1,
            "ping_time": 1000,
            "range_sample": -1,
        }
    ),
    target_variable="Sv",
    is_log=True,
    target_grid=target_grid,
)

aligned_ds_custom = ep.consolidate.add_depth(
    aligned_ds_custom
)

aligned_ds_custom.eshader.echogram(
    channel=selected_channel,
    vert_dim="depth",
).opts(
    cmap=ek500_cmap,
    clim=(-80, -30),
    colorbar=True,
    width=1000,
    height=450,
)


# Other visualization elements

The remaining examples use a processed Pacific hake survey dataset that includes `Sv`, `ping_time`, latitude, and longitude.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

work_dir = Path.cwd().resolve()
data_path = work_dir / "MVBS.zarr"
zip_path = work_dir / "MVBS.zip"

if not (data_path / "zarr.json").is_file():
    urlretrieve(
        "https://github.com/echostack-org/echoshader/"
        "releases/download/example-data-v1/MVBS.zip",
        zip_path,
    )
    with ZipFile(zip_path) as archive:
        archive.extractall(work_dir)

print("Dataset path:", data_path)
print("Metadata exists:", (data_path / "zarr.json").is_file())

assert (data_path / "zarr.json").is_file(), "Dataset extraction failed"

survey_MVBS = xr.open_zarr(data_path)
survey_MVBS

## Track

`eshader.track()` plots the geographic position associated with each ping. A moving platform is shown as a path; stationary data are represented as a point.


In [ ]:
survey_MVBS.eshader.track(
    tile="OpenTopoMap",
).opts(
    width=900,
    height=500,
)


In [ ]:
import geoviews as gv

track = survey_MVBS.eshader.track(
    tile="OpenTopoMap",
).opts(
    gv.opts.Path(
        color="red",
        line_width=2,
    ),
    gv.opts.Points(
        color="blue",
        size=8,
    ),
)

track

In [ ]:
channel_18 = survey_MVBS.channel.sel(
    channel=survey_MVBS.frequency_nominal == 18000
).item()

eg = survey_MVBS.eshader.echogram(
    channel=channel_18,
    vert_dim="depth",
).opts(
    cmap="jet",
)

pn.Column(track, eg)

## Histogram

`eshader.hist()` summarizes the `Sv` distribution by acoustic channel.


In [ ]:
survey_MVBS.eshader.hist(
    bins=24,
    overlay=True,
).opts(
    width=900,
    height=450,
)


## Statistics table

`eshader.table()` returns summary statistics for the complete dataset and individual channels.


In [ ]:
survey_MVBS.eshader.table()


## Curtain

`eshader.curtain()` projects the acoustic observations along the vessel track. The Plotly backend provides an interactive 3-D view without requiring additional application state.


In [ ]:
curtain_channel = survey_MVBS.channel.values[0]

survey_MVBS.eshader.curtain(
    channel=curtain_channel,
    ratio=0.001,
    engine="plotly",
    cmap="Viridis",
    clim=(-80, -30),
)

end of notebook

In [ ]:
from datetime import datetime, timezone
import echopype
import echoshader
import xarray
import panel
import holoviews

print(f"echopype: {echopype.__version__}")
print(f"echoshader: {echoshader.__version__}")
print(f"xarray: {xarray.__version__}")
print(f"panel: {panel.__version__}")
print(f"holoviews: {holoviews.__version__}")
print(f"Executed: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC}")